In [1]:
%matplotlib inline
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats
from scipy.optimize import curve_fit
import h5py

In [15]:
hf = h5py.File('uwphys434/gammaray_lab4.h5', 'r')

In [16]:
hf.keys()

<KeysViewHDF5 ['data']>

In [17]:
data = np.array(hf.get('data'))

In [18]:
print(np.shape(data))
data[:,0]

(4, 25920001)


array([9.40680016e+08, 3.15000000e+02, 4.50000000e+01, 1.00000000e+01])

In [19]:
hf.close()

In [20]:

time_gps = data[0, :]
solar_phase = data[1, :]
earth_long = data[2, :]
particle_counts = data[3, :]

print("---Data Summary---")
print(f"Total data points: {data.shape[1]:,}")
print(f"Time Range (GPS seconds): {time_gps.min()} to {time_gps.max()}")
print(f"Total Duration (days): {(time_gps.max() - time_gps.min()) / 86400:.2f}")
print(f"Solar Phase Range (deg): {solar_phase.min():.2f} to {solar_phase.max():.2f}")
print(f"Earth Longitude Range (deg): {earth_long.min():.2f} to {earth_long.max():.2f}")
print(f"Particle Counts - Min: {particle_counts.min()}, Max: {particle_counts.max()}, Mean: {particle_counts.mean():.2f}")



---Data Summary---
Total data points: 25,920,001
Time Range (GPS seconds): 940680016.0 to 943272016.0
Total Duration (days): 30.00
Solar Phase Range (deg): 0.00 to 360.00
Earth Longitude Range (deg): 0.00 to 359.99
Particle Counts - Min: 0.0, Max: 30.0, Mean: 7.10


##### 1

The columns are seconds (gps time), solar phase (deg), earth longitude, particle counts. There are ~25 million data points

A feature that stands out on the particle count distribution vs orbital coordinates chart is that the particle counts increase around the earth longitude 320 degrees and trail off around 360 degrees but remain elevated for some time. This is independent of the solar phase value, and has to do with the position of the satellite over the Earth. This is shown below in the particle count distribution vs satellite orbital coordinates graph. Furthermore, there is not much of a long term or slow time relation between count and days, rather the relation is in short intervals. From this, we know that during these short intervals the background increases from the position of the satellite as it goes through its orbit several times in a day. The background pdf will have two poisson distributions, one for the quiet region and another for the "hot" one where the background suddenly elevates and trails off.

Shown below the graph titled "background vs longitude" is a good visual representation of what is going on physically.

In [ ]:
# 2: how does the background change?
from scipy.stats import poisson

dt = np.median(np.diff(time_gps))                  # sample spacing
t_days = (time_gps - time_gps[0]) / 86400.0
lon = earth_long % 360
lon_idx = np.minimum(lon.astype(int), 359)         # 1-degree longitude bin of every sample
ph_idx  = np.minimum((solar_phase % 360).astype(int), 359)
counts  = particle_counts

# 1 periods hiding in the coordinates (instead of assuming 90 min)
def period_from_angle(ang_deg, n=200_000):
    a = np.unwrap(np.deg2rad(ang_deg[:n]))
    rate = np.polyfit(np.arange(n) * dt, a, 1)[0]  # rad / s
    return 2 * np.pi / abs(rate)
print(f"Longitude cycles every   {period_from_angle(earth_long)/60:9.2f} min")
print(f"Solar phase cycles every {period_from_angle(solar_phase)/86400:9.3f} days")

# 2 mean counts per 1-degree longitude bin and per 1-degree solar-phase bin 
def profile(idx, nb=360):
    n = np.bincount(idx, minlength=nb)
    s = np.bincount(idx, weights=counts, minlength=nb)
    m = s / np.maximum(n, 1)
    return m, np.sqrt(m / np.maximum(n, 1)), n    # mean, Poisson error on the mean, samples
lon_prof, lon_err, lon_n = profile(lon_idx)
ph_prof,  ph_err,  ph_n  = profile(ph_idx)

# 3) find the elevated region 
base, peak = np.median(lon_prof), lon_prof.max()
hot = lon_prof > 0.5 * (base + peak)               # True for bins above the half-way level
rise = np.where(hot & ~np.roll(hot, 1))[0]
fall = np.where(~hot & np.roll(hot, 1))[0]
print(f"\nBackground outside the hot zone : {lon_prof[~hot].mean():.3f} counts/100 ms")
print(f"Background inside the hot zone  : {lon_prof[hot].mean():.3f} counts/100 ms")
print(f"Hot zone runs from {rise[0]} deg to {fall[0]} deg longitude "
      f"(= {((rise[0]+180)%360)-180} to {((fall[0]+180)%360)-180} in -180..180 convention)")
print(f"Fraction of time spent in the hot zone: {hot[lon_idx].mean():.3f}")
in_zone = hot[lon_idx]

# 4) does the background also depend on solar phase or on slow time drift? 
# (use only samples outside the hot zone, so the hot zone's longitude coverage can't leak into the phase test)
n_o = np.bincount(ph_idx[~in_zone], minlength=360); s_o = np.bincount(ph_idx[~in_zone], weights=counts[~in_zone], minlength=360)
g = n_o > 100; m_o = s_o[g] / n_o[g]
chi2_ph = np.sum((m_o - counts[~in_zone].mean()) ** 2 / (counts[~in_zone].mean() / n_o[g]))
print(f"\nSolar-phase dependence (outside hot zone): chi2 = {chi2_ph:.0f} for {g.sum()} bins (chi2 ~ number of bins means NO dependence)")

nd = int(np.ceil(t_days.max()))                       # number of 1-day bins
day = np.minimum(t_days.astype(int), nd - 1)
def daily(mask):
    n = np.bincount(day[mask], minlength=nd); s = np.bincount(day[mask], weights=counts[mask], minlength=nd)
    return s / np.maximum(n, 1), np.sqrt(s / np.maximum(n, 1) / np.maximum(n, 1))
d_in, e_in = daily(in_zone); d_out, e_out = daily(~in_zone)
dd = np.arange(nd) + 0.5
for name, y, e in [("inside hot zone", d_in, e_in), ("outside hot zone", d_out, e_out)]:
    p, cov = np.polyfit(dd, y, 1, w=1 / e, cov="unscaled")
    print(f"Slope {name:17s}: {p[0]:+.5f} +/- {np.sqrt(cov[0,0]):.5f} counts per 100 ms per day")

#  5) is the background Poisson?  
out = counts[~in_zone]; inn = counts[in_zone]
print(f"\nVariance/mean outside zone: {out.var()/out.mean():.4f}   inside zone: {inn.var()/inn.mean():.4f}")

#  6) power spectrum of 1-minute counts: what periods are in the data? 
nmin = int(60 / dt); nblk = len(counts) // nmin
c_min = counts[: nblk * nmin].reshape(nblk, nmin).sum(1)
pw = np.abs(np.fft.rfft(c_min - c_min.mean())) ** 2
fr = np.fft.rfftfreq(nblk, d=1.0)                  # cycles per minute
top = np.argsort(pw[1:])[::-1][:4] + 1
print("Strongest periodicities (minutes):", np.round(1 / fr[top], 1))

# --- plots ---
fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0,0].errorbar(np.arange(360) + .5, lon_prof, lon_err, fmt='.', ms=3, color='k')
ax[0,0].axhline(base, color='b', ls=':'); ax[0,0].set(xlabel='Earth longitude (deg)', ylabel='mean counts / 100 ms', title='Background vs longitude')
ax[0,1].errorbar(np.arange(360) + .5, ph_prof, ph_err, fmt='.', ms=3, color='k')
ax[0,1].set(xlabel='Solar phase (deg)', ylabel='mean counts / 100 ms', title='Background vs solar phase', ylim=ax[0,0].get_ylim())
ax[1,0].errorbar(dd, d_in, e_in, fmt='o', label='inside hot zone')
ax[1,0].errorbar(dd, d_out, e_out, fmt='s', label='outside hot zone')
ax[1,0].set(xlabel='day', ylabel='mean counts / 100 ms', title='Time dependence over days'); ax[1,0].legend()
# mean counts in 5x5 degree cells (a far cleaner picture than the overplotted scatter)
H = np.histogram2d(lon, solar_phase % 360, bins=72, range=[[0,360],[0,360]], weights=counts)[0] / \
    np.maximum(np.histogram2d(lon, solar_phase % 360, bins=72, range=[[0,360],[0,360]])[0], 1)
im = ax[1,1].imshow(H.T, origin='lower', extent=[0,360,0,360], aspect='auto', cmap='viridis')
plt.colorbar(im, ax=ax[1,1], label='mean counts'); ax[1,1].set(xlabel='Earth longitude (deg)', ylabel='Solar phase (deg)', title='Particle count distribution vs satellite orbital coordinates')
plt.tight_layout(); plt.show()

###### 2 

The background changes most in relation to longitude, with little fluctuation from solar phase or time in days. Because of this relation to longitude and the fact that the satellite is orbiting, this also will correspond to a relationship between background count and time that shows on a scale of minutes (shown below). The fluctuation is in periods of about an hour or so.

In [ ]:
# Q3: background model with time and longitude dependence
from scipy.optimize import minimize

# Model: lambda(lon, t) = [b0 + b1*t] + [s0 + s1*t] * S(lon)
#   S(lon) = smooth "box" centred at c, half-width hw, edge sharpness w (periodic in longitude)
def saa_shape(lon_deg, c, hw, w):
    w = np.maximum(np.abs(w), 0.3)                    # edges can't be sharper than the 1-degree binning can resolve
    phi = (lon_deg - c + 180.0) % 360.0 - 180.0
    return 0.5 * (np.tanh((phi + hw) / w) - np.tanh((phi - hw) / w))

def bg_model(t_days, lon_deg, b0, b1, s0, s1, c, hw, w):
    return (b0 + b1 * t_days) + (s0 + s1 * t_days) * saa_shape(lon_deg, c, hw, w)

# --- bin data into (6-hour) x (1-degree longitude) cells
t_bin_sec = 6 * 3600
nT = int(np.ceil((time_gps[-1] - time_gps[0]) / t_bin_sec))
ti = np.minimum(((time_gps - time_gps[0]) // t_bin_sec).astype(int), nT - 1)
cell = ti * 360 + lon_idx
N = np.bincount(cell, minlength=nT * 360).reshape(nT, 360).astype(float)               # samples per cell
K = np.bincount(cell, weights=counts, minlength=nT * 360).reshape(nT, 360)             # total counts per cell
Tg, Lg = np.meshgrid((np.arange(nT) + 0.5) * t_bin_sec / 86400.0, np.arange(360) + 0.5, indexing='ij')
ok = N > 0

def nll(p):                                           # Poisson negative log-likelihood (dropping constants)
    lam = bg_model(Tg, Lg, *p)
    if np.any(lam[ok] <= 0): return 1e30
    return np.sum(N[ok] * lam[ok] - K[ok] * np.log(lam[ok])) / N.sum()   

# initial guesses from the Q2 profile
width0 = (fall[0] - rise[0]) % 360
p0 = [lon_prof[~hot].mean(), 0.0, lon_prof[hot].mean() - lon_prof[~hot].mean(), 0.0,
      (rise[0] + width0 / 2) % 360, width0 / 2, 1.0]
bounds = [(0, 20), (-1, 1), (-20, 40), (-1, 1), (0, 360), (5, 180), (0.3, 30)]
res = minimize(nll, p0, method='L-BFGS-B', bounds=bounds, options=dict(ftol=1e-15, gtol=1e-10, maxiter=2000))
res = minimize(nll, res.x, method='Nelder-Mead', options=dict(xatol=1e-6, fatol=1e-14, maxiter=5000))   # polish
popt = res.x

def num_hessian(f, x, rel=1e-3):                       
    n = len(x); h = rel * np.maximum(np.abs(x), 1e-2); H = np.zeros((n, n))
    for i in range(n):
        for j in range(i, n):
            xpp = x.copy(); xpp[i] += h[i]; xpp[j] += h[j]
            xpm = x.copy(); xpm[i] += h[i]; xpm[j] -= h[j]
            xmp = x.copy(); xmp[i] -= h[i]; xmp[j] += h[j]
            xmm = x.copy(); xmm[i] -= h[i]; xmm[j] -= h[j]
            H[i, j] = H[j, i] = (f(xpp) - f(xpm) - f(xmp) + f(xmm)) / (4 * h[i] * h[j])
    return H
names = ['b0 (quiet region baseline)', 'b1 (quiet region drift /day)', 's0 (hot zone excess)', 's1 (hot zone drift /day)',
         'c  (hot zone centre, deg)', 'hw (hot zone half-width, deg)', 'w  (edge sharpness, deg)']
try:
    perr = np.sqrt(np.diag(np.linalg.inv(num_hessian(nll, popt)) / N.sum()))   # nll is per-sample, so divide by N_total
except Exception:
    perr = np.full(len(popt), np.nan)
print("--- Best-fit parameters (maximum Poisson likelihood) ---")
for n_, v, e in zip(names, popt, perr):
    print(f"{n_:28s} {v:10.4f} +/- {e:.4f}")

#  how good is it? 
lam_cell = bg_model(Tg, Lg, *popt)
E = N * lam_cell
def deviance(Kc, Ec):                                 
    m = ok & (Kc > 0)
    return 2 * (np.sum(Kc[m] * np.log(Kc[m] / Ec[m])) - np.sum(Kc[ok] - Ec[ok]))
D_model = deviance(K, E)
dof = ok.sum() - len(popt)
E_const = N * (K.sum() / N.sum())                   
D_const = deviance(K, E_const)
print(f"\nDeviance / dof, constant-rate model : {D_const/(ok.sum()-1):10.2f}   ")
print(f"Deviance / dof, physical model           : {D_model/dof:10.3f}   (1.0 =  perfect)")
pull = np.where(ok, (K - E) / np.sqrt(np.maximum(E, 1e-9)), np.nan)
print(f"Pull mean/std (want 0 / 1): {np.nanmean(pull):+.3f} / {np.nanstd(pull):.3f}")

# sample-level check: predicted vs observed distribution of the raw counts 
kmax = int(counts.max())
kk = np.arange(kmax + 1)[:, None, None]
exp_k = (N[None] * poisson.pmf(kk, lam_cell[None])).sum((1, 2))
obs_k = np.bincount(counts.astype(int), minlength=kmax + 1)

fig, ax = plt.subplots(2, 2, figsize=(13, 8))
ax[0,0].semilogy(kk.ravel(), obs_k, 'ko', label='data'); ax[0,0].semilogy(kk.ravel(), exp_k, 'r-', label='model (Poisson mixture)')
ax[0,0].set(xlabel='counts per 100 ms', ylabel='number of samples', title='Count distribution: data vs model'); ax[0,0].legend()
m = exp_k > 5
ax[0,1].plot(kk.ravel()[m], (obs_k[m] - exp_k[m]) / np.sqrt(exp_k[m]), 'ko-')
ax[0,1].axhline(0, color='r'); ax[0,1].set(xlabel='counts per 100 ms', ylabel='(obs - exp)/sqrt(exp)', title='Residuals of the count distribution (bins with exp>5)')
ax[1,0].plot(np.arange(360) + .5, np.nanmean(pull, axis=0), 'k.'); ax[1,0].axhline(0, color='r')
ax[1,0].set(xlabel='Earth longitude (deg)', ylabel='mean pull', title='Residual pull vs longitude')
ax[1,1].hist(pull[ok], bins=60, density=True, alpha=.6, label='pulls')
xs = np.linspace(-5, 5, 200); ax[1,1].plot(xs, stats.norm.pdf(xs), 'r', label='N(0,1)'); ax[1,1].legend()
ax[1,1].set(xlabel='(K-E)/sqrt(E) per cell', title='Pull distribution')
plt.tight_layout(); plt.show()

# direct data-vs-model picture: three hours of data at 1-minute resolution 
sel = (time_gps - time_gps[0]) < 3 * 3600
nm = sel.sum() // nmin
lam_sel = bg_model(t_days[sel], lon[sel], *popt)
plt.figure(figsize=(13, 4))
plt.plot(c_min[:nm] / nmin, 'k', lw=1, label='data (1-min mean)')
plt.plot(lam_sel[: nm * nmin].reshape(nm, nmin).mean(1), 'r', lw=1.5, label='model')
plt.xlabel('minutes from start'); plt.ylabel('mean counts / 100 ms'); plt.legend(); plt.title('First 3 hours: data vs model'); plt.show()

The model captures the "hot zone" to some degree but statistically isn't great since it doesn't get the tapering off that is visible in the data very well. There are sections where the model over-estimates right after the background peaks and then underestimates when the data is still elevated. The model gets the average level and the location of the zone but isn't the best approximation of its shape.

In [ ]:
# Q4: 5-sigma threshold for a 100 ms burst, vs position/time
p5 = stats.norm.sf(5)                                  # one-sided 5 sigma = 2.87e-7

def n_required(lam, p=p5):
    """Smallest integer N with P(X >= N | background lam) < p   (exact Poisson tail)."""
    lam = np.atleast_1d(lam).astype(float)
    out = np.empty(lam.shape, dtype=int)
    for i, l in enumerate(lam.ravel()):
        n = int(poisson.isf(p, l)) + 1
        while poisson.sf(n - 1, l) >= p: n += 1        # P(X>=n) = sf(n-1) must be below p
        while n > 0 and poisson.sf(n - 2, l) < p: n -= 1
        out.ravel()[i] = n
    return out

# threshold for every cell, then summarise how much of the data lives at each threshold
uniq, inv = np.unique(np.round(lam_cell, 3), return_inverse=True)
thr_cell = n_required(uniq)[inv].reshape(lam_cell.shape)

print("--- 5-sigma thresholds in a 100 ms window ---")
print(f"Model background ranges from {lam_cell[ok].min():.2f} to {lam_cell[ok].max():.2f} counts/100 ms")
for t in np.unique(thr_cell[ok]):
    frac = N[ok & (thr_cell == t)].sum() / N[ok].sum()
    l_here = lam_cell[ok & (thr_cell == t)]
    print(f"  need >= {t:3d} counts   (background {l_here.min():5.2f}-{l_here.max():5.2f})   applies {100*frac:5.1f}% of the time")

# the quiet vs hot-zone numbers, evaluated at the middle of the run
t_mid = 0.5 * t_days.max()
lam_quiet = bg_model(t_mid, popt[4] + 180, *popt)      # a longitude on the opposite side of the Earth
lam_hot   = bg_model(t_mid, popt[4], *popt)            # centre of the hot zone
for label, l in [("quiet (outside hot zone)", lam_quiet), ("hot zone", lam_hot)]:
    n5 = n_required(l)[0]
    print(f"{label:26s} background {l:6.3f} -> need {n5} counts = {n5-l:5.1f} above background "
          f"= {(n5-l)/np.sqrt(l):4.1f} Gaussian sigma of the background")

# trials factor 
ntrials = len(counts)
p_global = p5 / ntrials
print(f"\n{ntrials:,} windows -> expect {ntrials*p5:.1f} false 5-sigma alarms in this dataset from background alone.")
print(f"To get a 5-sigma GLOBAL result you need p < {p_global:.1e} per window: "
      f"quiet -> {n_required(lam_quiet, p_global)[0]} counts, hot zone -> {n_required(lam_hot, p_global)[0]} counts")

# check against the data: how many windows exceed their local threshold?
thr_s = np.where(in_zone, n_required(lam_hot)[0], n_required(lam_quiet)[0])
n_exc = int(np.sum(counts >= thr_s))
print(f"Windows above their local 5-sigma threshold: {n_exc} (expected from background alone: {ntrials*p5:.1f})")

# plot: threshold as a function of longitude (and the background it comes from)
lon_c = np.arange(360) + 0.5
lam_lon = bg_model(t_mid, lon_c, *popt)
thr_lon = n_required(np.round(lam_lon, 3))
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(lon_c, lam_lon, 'k'); ax[0].set(xlabel='Earth longitude (deg)', ylabel='model background (counts/100 ms)', title='Background model')
ax[1].step(lon_c, thr_lon, 'r', where='mid'); ax[1].set(xlabel='Earth longitude (deg)', ylabel='counts needed in 100 ms', title='5-sigma discovery threshold')
for a in ax: a.grid(alpha=.3)
plt.tight_layout(); plt.show()

###### 4. 

While the satellite is in the quiet zone, 23 counts are required for a 5-sigma discovery and 29 are required for the 5-sigma threshold when in the hot zone.

In [ ]:
# Q5: compare the fitted zone with the South Atlantic Anomaly
def to_pm180(x): return (x + 180) % 360 - 180
lo, hi = to_pm180(popt[4] - popt[5]), to_pm180(popt[4] + popt[5])
print(f"Fitted hot zone: {lo:.1f} to {hi:.1f} deg longitude (centre {to_pm180(popt[4]):.1f}, half-width {popt[5]:.1f})")
print("Real world South Atlantic Anomaly span according to wikipedia: roughly -90 to +40 deg longitude")

x = np.linspace(-180, 180, 721)
plt.figure(figsize=(10, 4))
plt.plot(x, bg_model(t_mid, x % 360, *popt), 'r', label='fitted background model')
plt.axvspan(-90, 40, color='orange', alpha=.2, label='typical SAA extent')
plt.xlabel('Longitude (deg, -180..180)'); plt.ylabel('counts / 100 ms'); plt.legend(); plt.grid(alpha=.3)
plt.title('Fitted hot zone vs South Atlantic Anomaly'); plt.show()

##### 5. 

The variable background appears to be like the effect that the South Atlantic Anomaly has on satellites orbiting the Earth, where the inner van allen radiation belt comes down close to Earth due to the magnetic field being weaker around the south Atlantic and it contains charged particles that can hit the detector on the satellite, causing this elevated background that fluctuates based on position over the Earth's surface. In the case of the south atlantic anomaly, the span is from -90 degrees to +40 degrees in longitude shown above. Here in this hypothetical, we can see the background increases the most around 320 degrees to about 50 degrees longitude or so (in red).